# 🏗️ ETF Portfolio Backtesting System - Integrated System

**Main Controller Notebook - ผู้ใช้โต้ตอบผ่าน Notebook นี้เท่านั้น**

## 🎯 System Architecture:

```
Main Controller (Notebook นี้)
    ├── System Configuration
    ├── Module Loader
    ├── Portfolio Management → CRUD Module
    ├── ETF Management → CRUD Module  
    ├── Backtesting → Backtesting Module
    ├── Analytics → Analytics Module
    └── Database → MySQL
```

## 📋 วิธีใช้งาน:
1. **Run Step 1:** Setup & Configuration
2. **Run Step 2:** Initialize System & Load Modules
3. **Run Step 3+:** ใช้งานผ่าน Interactive Dashboard

---

# Step 1: System Configuration

⚠️ **แก้ MySQL password ตรงนี้**

In [ ]:
# System Configuration
import os
import sys
from pathlib import Path

# Database Configuration
DB_CONFIG = {
    'host': '127.0.0.1',
    'port': 3306,
    'user': 'root',
    'password': 'krittanut123456',  # ⚠️ แก้ตรงนี้!
    'database': 'etf_backtesting'
}

# Project Directory
PROJECT_DIR = Path(os.getcwd()).absolute()

print("✅ System Configuration Loaded")
print(f"   Project Dir: {PROJECT_DIR}")
print(f"   Database: {DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}")

# Test Connection
import mysql.connector
try:
    conn = mysql.connector.connect(**DB_CONFIG)
    cursor = conn.cursor()
    cursor.execute("SELECT VERSION()")
    version = cursor.fetchone()[0]
    cursor.close()
    conn.close()
    print(f"\n✅ MySQL Connected! (Version: {version})")
except Exception as e:
    print(f"\n❌ Connection Failed: {e}")

# Step 2: Initialize System & Load All Modules

**Main Controller จะ load ทุก subsystem modules**

In [ ]:
import importlib.util
import mysql.connector
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets

# ===================================================================
# MAIN CONTROLLER CLASS
# ===================================================================

class MainController:
    """Main System Controller - Integrated System"""
    
    def __init__(self, db_config, project_dir):
        self.db_config = db_config
        self.project_dir = project_dir
        self.modules = {}
        self.status = "Initializing..."
        
    def load_module(self, module_name, file_path):
        """Load a Python module dynamically"""
        try:
            full_path = self.project_dir / file_path
            if not full_path.exists():
                print(f"⚠️  Module not found: {file_path}")
                return False
            
            spec = importlib.util.spec_from_file_location(module_name, full_path)
            module = importlib.util.module_from_spec(spec)
            sys.modules[module_name] = module
            spec.loader.exec_module(module)
            
            self.modules[module_name] = module
            print(f"✓ Loaded: {module_name}")
            return True
        except Exception as e:
            print(f"✗ Error loading {module_name}: {e}")
            return False
    
    def initialize(self):
        """Initialize all subsystem modules"""
        print("\n" + "="*80)
        print("🚀 INITIALIZING INTEGRATED SYSTEM")
        print("="*80)
        print("\n📦 Loading Subsystem Modules...\n")
        
        modules_to_load = [
            ('crud_operations', 'crud_operations/crud_operations.py'),
            ('backtesting_engine', 'backtesting/backtesting_engine.py'),
            ('analytics', 'analytics/analytics.py'),
        ]
        
        success_count = 0
        for name, path in modules_to_load:
            if self.load_module(name, path):
                success_count += 1
        
        print("\n" + "="*80)
        print(f"✅ Loaded {success_count}/{len(modules_to_load)} modules")
        print("="*80)
        
        if success_count == len(modules_to_load):
            self.status = "Ready"
            print("\n✅ System initialized successfully!")
            print("\n📊 Available Modules:")
            for name in self.modules.keys():
                print(f"   • {name}")
            return True
        else:
            self.status = "Partial"
            print("\n⚠️  Some modules failed to load")
            return False
    
    def get_module(self, name):
        """Get loaded module"""
        return self.modules.get(name)
    
    def get_connection(self):
        """Get database connection"""
        try:
            return mysql.connector.connect(**self.db_config)
        except Exception as e:
            print(f"❌ Connection error: {e}")
            return None

# ===================================================================
# CREATE MAIN CONTROLLER INSTANCE
# ===================================================================

controller = MainController(DB_CONFIG, PROJECT_DIR)
controller.initialize()

print("\n" + "="*80)
print("🎉 Main Controller Ready!")
print("="*80)
print("\nYou can now use the Interactive Dashboard below.")

---

# 🎛️ Interactive Dashboard - Main Controller

**ใช้งานระบบผ่าน Dashboard นี้**

---

## 📊 Section 1: System Overview

In [ ]:
def show_system_status():
    """Show system status"""
    clear_output(wait=True)
    
    print("\n" + "="*80)
    print("📊 SYSTEM STATUS")
    print("="*80)
    print(f"Status: {controller.status}")
    print(f"Project Dir: {controller.project_dir}")
    print(f"Database: {controller.db_config['database']}")
    print(f"\nLoaded Modules: {len(controller.modules)}")
    for name in controller.modules.keys():
        print(f"  ✓ {name}")
    
    # Database stats
    try:
        conn = controller.get_connection()
        cursor = conn.cursor()
        
        tables = ['etfs', 'portfolios', 'daily_prices', 'backtests']
        print(f"\n📈 Database Statistics:")
        for table in tables:
            cursor.execute(f"SELECT COUNT(*) FROM {table}")
            count = cursor.fetchone()[0]
            print(f"  {table:20s}: {count:,}")
        
        cursor.close()
        conn.close()
    except Exception as e:
        print(f"\n❌ Error fetching stats: {e}")
    
    print("="*80)

# Button to show status
status_btn = widgets.Button(description="🔍 Show System Status", button_style='info')
status_output = widgets.Output()

def on_status_clicked(b):
    with status_output:
        show_system_status()

status_btn.on_click(on_status_clicked)

display(status_btn, status_output)

## 📁 Section 2: Portfolio Management (CRUD Module)

In [ ]:
def view_portfolios():
    """View all portfolios - calls CRUD module"""
    clear_output(wait=True)
    
    print("\n" + "="*80)
    print("📁 PORTFOLIO MANAGEMENT")
    print("📊 Calling CRUD Module: get_all_portfolios()")
    print("="*80 + "\n")
    
    crud = controller.get_module('crud_operations')
    if not crud:
        print("❌ CRUD module not available!")
        return
    
    try:
        portfolios = crud.get_all_portfolios(controller.db_config)
        if portfolios:
            df = pd.DataFrame(portfolios, columns=['ID', 'Name', 'Description', 'Created'])
            display(df)
            print(f"\n✅ Total: {len(portfolios)} portfolios")
        else:
            print("⚠️  No portfolios found")
    except Exception as e:
        print(f"❌ Error: {e}")

def view_portfolio_details(portfolio_id):
    """View portfolio details - calls CRUD module"""
    clear_output(wait=True)
    
    print("\n" + "="*80)
    print(f"📁 PORTFOLIO DETAILS (ID: {portfolio_id})")
    print(f"📊 Calling CRUD Module: get_portfolio_details({portfolio_id})")
    print("="*80 + "\n")
    
    crud = controller.get_module('crud_operations')
    if not crud:
        print("❌ CRUD module not available!")
        return
    
    try:
        details = crud.get_portfolio_details(portfolio_id, controller.db_config)
        if details:
            print(f"Name: {details[0][1]}")
            print(f"Description: {details[0][2]}\n")
            
            # Get ETFs in portfolio
            conn = controller.get_connection()
            query = f"""
            SELECT pe.ticker, e.name, e.category, pe.weight
            FROM portfolio_etfs pe
            JOIN etfs e ON pe.ticker = e.ticker
            WHERE pe.portfolio_id = {portfolio_id}
            ORDER BY pe.weight DESC
            """
            df = pd.read_sql(query, conn)
            conn.close()
            
            if not df.empty:
                display(df)
                print(f"\nTotal Weight: {df['weight'].sum():.2f}%")
        else:
            print(f"❌ Portfolio {portfolio_id} not found")
    except Exception as e:
        print(f"❌ Error: {e}")

# Portfolio Management UI
portfolio_action = widgets.Dropdown(
    options=['View All Portfolios', 'View Portfolio Details'],
    description='Action:',
    style={'description_width': '150px'}
)

portfolio_id_input = widgets.IntText(
    value=1,
    description='Portfolio ID:',
    style={'description_width': '150px'}
)

portfolio_btn = widgets.Button(description="▶️ Execute", button_style='success')
portfolio_output = widgets.Output()

def on_portfolio_clicked(b):
    with portfolio_output:
        if portfolio_action.value == 'View All Portfolios':
            view_portfolios()
        elif portfolio_action.value == 'View Portfolio Details':
            view_portfolio_details(portfolio_id_input.value)

portfolio_btn.on_click(on_portfolio_clicked)

portfolio_ui = widgets.VBox([
    widgets.HBox([portfolio_action, portfolio_id_input, portfolio_btn]),
    portfolio_output
])

display(portfolio_ui)

## 📊 Section 3: ETF Data Management (CRUD Module)

In [ ]:
def view_etfs(category=None):
    """View ETFs - calls CRUD module"""
    clear_output(wait=True)
    
    print("\n" + "="*80)
    print("📊 ETF DATA MANAGEMENT")
    print("📊 Calling CRUD Module: get_all_etfs()")
    print("="*80 + "\n")
    
    crud = controller.get_module('crud_operations')
    if not crud:
        print("❌ CRUD module not available!")
        return
    
    try:
        etfs = crud.get_all_etfs(controller.db_config)
        if etfs:
            df = pd.DataFrame(etfs, columns=['Ticker', 'Name', 'Category', 'Expense Ratio', 'Inception', 'Created'])
            
            if category and category != 'All':
                df = df[df['Category'] == category]
            
            display(df)
            print(f"\n✅ Total: {len(df)} ETFs")
        else:
            print("⚠️  No ETFs found")
    except Exception as e:
        print(f"❌ Error: {e}")

# Get categories for dropdown
try:
    conn = controller.get_connection()
    cats_df = pd.read_sql("SELECT DISTINCT category FROM etfs ORDER BY category", conn)
    conn.close()
    categories = ['All'] + cats_df['category'].tolist()
except:
    categories = ['All']

etf_category = widgets.Dropdown(
    options=categories,
    description='Category:',
    style={'description_width': '150px'}
)

etf_btn = widgets.Button(description="▶️ View ETFs", button_style='success')
etf_output = widgets.Output()

def on_etf_clicked(b):
    with etf_output:
        category = etf_category.value if etf_category.value != 'All' else None
        view_etfs(category)

etf_btn.on_click(on_etf_clicked)

etf_ui = widgets.VBox([
    widgets.HBox([etf_category, etf_btn]),
    etf_output
])

display(etf_ui)

## 💹 Section 4: Price Data Analysis

In [ ]:
def view_price_data(ticker, limit):
    """View price data"""
    clear_output(wait=True)
    
    print("\n" + "="*80)
    print(f"💹 PRICE DATA: {ticker} (Latest {limit} days)")
    print("="*80 + "\n")
    
    try:
        conn = controller.get_connection()
        query = f"""
        SELECT date, close, volume
        FROM daily_prices
        WHERE ticker = '{ticker}'
        ORDER BY date DESC
        LIMIT {limit}
        """
        df = pd.read_sql(query, conn)
        conn.close()
        
        if df.empty:
            print(f"❌ No data found for {ticker}")
        else:
            display(df)
            print(f"\n✅ Loaded {len(df)} records")
    except Exception as e:
        print(f"❌ Error: {e}")

# Get tickers for dropdown
try:
    conn = controller.get_connection()
    tickers_df = pd.read_sql("SELECT DISTINCT ticker FROM etfs ORDER BY ticker", conn)
    conn.close()
    tickers = tickers_df['ticker'].tolist()
except:
    tickers = ['SPY']

price_ticker = widgets.Dropdown(
    options=tickers,
    value='SPY',
    description='Ticker:',
    style={'description_width': '150px'}
)

price_limit = widgets.IntSlider(
    value=10,
    min=5,
    max=100,
    step=5,
    description='Days:',
    style={'description_width': '150px'}
)

price_btn = widgets.Button(description="▶️ View Prices", button_style='success')
price_output = widgets.Output()

def on_price_clicked(b):
    with price_output:
        view_price_data(price_ticker.value, price_limit.value)

price_btn.on_click(on_price_clicked)

price_ui = widgets.VBox([
    widgets.HBox([price_ticker, price_limit, price_btn]),
    price_output
])

display(price_ui)

## 🔬 Section 5: Backtesting (Backtesting Module)

In [ ]:
def run_backtest(portfolio_id, strategy, start_date, end_date, capital):
    """Run backtest - calls Backtesting module"""
    clear_output(wait=True)
    
    print("\n" + "="*80)
    print("🔬 BACKTESTING ENGINE")
    print(f"📊 Calling Backtesting Module: run_backtest()")
    print("="*80 + "\n")
    print(f"Portfolio ID: {portfolio_id}")
    print(f"Strategy: {strategy}")
    print(f"Period: {start_date} to {end_date}")
    print(f"Capital: ${capital:,.2f}")
    print("\nRunning backtest... Please wait...\n")
    
    backtest = controller.get_module('backtesting_engine')
    if not backtest:
        print("❌ Backtesting module not available!")
        return
    
    try:
        backtest_id = backtest.run_backtest(
            portfolio_id=portfolio_id,
            start_date=start_date,
            end_date=end_date,
            initial_capital=capital,
            strategy=strategy,
            db_config=controller.db_config
        )
        
        if backtest_id:
            print(f"✅ Backtest completed! ID: {backtest_id}")
            
            # Show results
            conn = controller.get_connection()
            query = f"""
            SELECT final_value, total_return
            FROM backtests
            WHERE backtest_id = {backtest_id}
            """
            df = pd.read_sql(query, conn)
            conn.close()
            
            if not df.empty:
                print(f"\nFinal Value: ${df['final_value'].values[0]:,.2f}")
                print(f"Total Return: {df['total_return'].values[0]:.2f}%")
        else:
            print("❌ Backtest failed!")
    except Exception as e:
        print(f"❌ Error: {e}")

backtest_portfolio = widgets.IntText(
    value=1,
    description='Portfolio ID:',
    style={'description_width': '150px'}
)

backtest_strategy = widgets.Dropdown(
    options=['buy_hold', 'rebalancing_monthly', 'dca'],
    description='Strategy:',
    style={'description_width': '150px'}
)

backtest_start = widgets.Text(
    value='2020-01-01',
    description='Start Date:',
    style={'description_width': '150px'}
)

backtest_end = widgets.Text(
    value='2024-12-31',
    description='End Date:',
    style={'description_width': '150px'}
)

backtest_capital = widgets.FloatText(
    value=100000.0,
    description='Capital:',
    style={'description_width': '150px'}
)

backtest_btn = widgets.Button(description="▶️ Run Backtest", button_style='warning')
backtest_output = widgets.Output()

def on_backtest_clicked(b):
    with backtest_output:
        run_backtest(
            backtest_portfolio.value,
            backtest_strategy.value,
            backtest_start.value,
            backtest_end.value,
            backtest_capital.value
        )

backtest_btn.on_click(on_backtest_clicked)

backtest_ui = widgets.VBox([
    backtest_portfolio,
    backtest_strategy,
    backtest_start,
    backtest_end,
    backtest_capital,
    backtest_btn,
    backtest_output
])

display(backtest_ui)

## 📈 Section 6: Analytics & Insights (Analytics Module)

In [ ]:
def run_analytics(insight_type, portfolio_ids):
    """Run analytics - calls Analytics module"""
    clear_output(wait=True)
    
    print("\n" + "="*80)
    print("📈 ANALYTICS MODULE")
    print(f"📊 Calling Analytics Module: generate_insight{insight_type}()")
    print("="*80 + "\n")
    
    analytics = controller.get_module('analytics')
    if not analytics:
        print("❌ Analytics module not available!")
        return
    
    try:
        if insight_type == '1':
            print("Running Insight 1: Risk-Adjusted Performance Analysis...")
            print("This may take a few minutes...\n")
            
            analytics.generate_insight1(
                portfolio_ids=portfolio_ids,
                start_date='2020-01-01',
                end_date='2024-12-31',
                benchmark='SPY',
                db_config=controller.db_config
            )
            
            print("\n✅ Analysis completed!")
            print("📄 Report saved to: insight1_risk_adjusted_report.txt")
            
        elif insight_type == '2':
            print("Running Insight 2: Optimal Rebalancing Frequency...")
            print("This will take 3-5 minutes...\n")
            
            analytics.generate_insight2(
                portfolio_id=portfolio_ids[0],
                start_date='2020-01-01',
                end_date='2024-12-31',
                initial_capital=100000.0,
                db_config=controller.db_config
            )
            
            print("\n✅ Analysis completed!")
            print("📄 Report saved to: insight2_rebalancing_analysis.txt")
            
        elif insight_type == '3':
            print("Running Insight 3: DCA vs Lump Sum Analysis...")
            print("This may take a few minutes...\n")
            
            analytics.generate_insight3(
                portfolio_id=portfolio_ids[0],
                total_capital=100000.0,
                investment_months=12,
                start_date='2020-01-01',
                end_date='2024-12-31',
                db_config=controller.db_config
            )
            
            print("\n✅ Analysis completed!")
            print("📄 Report saved to: insight3_dca_vs_lumpsum.txt")
    
    except Exception as e:
        print(f"❌ Error: {e}")

analytics_insight = widgets.Dropdown(
    options=['1 - Risk-Adjusted Performance', '2 - Optimal Rebalancing', '3 - DCA vs Lump Sum'],
    description='Insight:',
    style={'description_width': '150px'}
)

analytics_portfolios = widgets.Text(
    value='1,2,3',
    description='Portfolio IDs:',
    placeholder='1,2,3',
    style={'description_width': '150px'}
)

analytics_btn = widgets.Button(description="▶️ Run Analytics", button_style='danger')
analytics_output = widgets.Output()

def on_analytics_clicked(b):
    with analytics_output:
        insight_num = analytics_insight.value[0]
        portfolio_ids = [int(x.strip()) for x in analytics_portfolios.value.split(',')]
        run_analytics(insight_num, portfolio_ids)

analytics_btn.on_click(on_analytics_clicked)

analytics_ui = widgets.VBox([
    analytics_insight,
    analytics_portfolios,
    analytics_btn,
    analytics_output
])

display(analytics_ui)

---

# 📊 Summary

## ✅ Integrated System Features:

1. **Main Controller** - ควบคุมทุก modules
2. **Dynamic Module Loading** - Load subsystems อัตโนมัติ
3. **Interactive Dashboard** - UI ที่ใช้งานง่าย
4. **Module Integration:**
   - Portfolio Management → CRUD Module
   - ETF Management → CRUD Module
   - Price Analysis → Database Module
   - Backtesting → Backtesting Module
   - Analytics → Analytics Module

## 🎯 User Workflow:

```
User → Interactive Dashboard → Main Controller → Subsystem Module → Database
```

## 📚 Documentation:

- **INTEGRATED_SYSTEM_GUIDE.md** - Complete architecture guide
- **USER_GUIDE_TH.md** - User manual
- **HOW_TO_RUN.md** - Quick start guide

---

**ผู้ใช้โต้ตอบผ่าน Notebook นี้เท่านั้น - Main Controller จัดการทุกอย่าง!** 🎯

---